# 121 — Structured Generation with Outlines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/121-structured-generation-outlines/structured_generation_outlines_workbook.ipynb)

This workbook demonstrates two approaches to guaranteed structured output from LLMs:

1. **outlines** — modifies the token sampling process (CFG) so invalid tokens are impossible. One pass, no retries.
2. **instructor** — wraps OpenAI tool-calling with Pydantic validation and retries until the schema is satisfied.

**Key insight:** outlines works at the *decoding* level; instructor works at the *prompt/retry* level.

In [ ]:
# Setup — install dependencies
# Note: outlines + transformers require a GPU and ~2GB download.
# The code will gracefully fall back if they are not available.
!pip install -q openai pydantic python-dotenv
!pip install -q instructor  # optional but recommended for comparison
# Uncomment below for full outlines support (GPU required):
# !pip install -q outlines transformers torch

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Set your OpenAI API key
# os.environ['OPENAI_API_KEY'] = 'sk-...'
print('API key set:', bool(os.environ.get('OPENAI_API_KEY')))

## Part 1: The Structured Output Problem

When you ask an LLM to return JSON, it can fail in subtle ways:

```
Prompt: "Return JSON with name, age, occupation"
LLM:    "Sure! Here is the JSON: {\"name\": \"Alice\", \"age\": \"thirty-two\", \"ocup\": ...}"
                                                            ^^^^^^^^^^^            ^^^^
                                                          wrong type          typo in key
```

**Two solutions:**
- **Constrained decoding (outlines):** Make invalid tokens impossible during sampling
- **Retry loops (instructor):** Validate output, send errors back, retry until valid

## Part 2: Two Approaches Side by Side

| | outlines | instructor |
|---|---|---|
| **Mechanism** | CFG sampling (token masking) | Tool-calling + Pydantic retry |
| **API calls** | 1 (local model) | 1–3+ (hosted API) |
| **Guarantee** | Mathematical — impossible to fail | Probabilistic — retries until valid |
| **Best for** | Local models (HuggingFace) | Hosted APIs (OpenAI, Anthropic) |
| **Requires GPU** | Yes (for local models) | No |
| **Schema support** | JSON, regex, choice, CFG grammar | Pydantic v1/v2, full nesting |

## How CFG Sampling Works

```
Normal sampling:
  token probabilities → sample from all tokens → output token

Outlines CFG sampling:
  token probabilities → MASK invalid tokens (p=0) → sample from valid tokens → output token
                              ^
                    determined by JSON schema / regex / grammar
```

At each step, outlines computes which tokens are valid given the schema state machine,
sets all other token logits to `-inf`, and samples normally. The output **must** be valid.

## Part 3: Define Pydantic Models

In [ ]:
from pydantic import BaseModel

class ExtractedPerson(BaseModel):
    name: str
    age: int
    occupation: str

class ParsedDate(BaseModel):
    year: int
    month: int
    day: int

# View the JSON schema that outlines will enforce
import json
print('ExtractedPerson schema:')
print(json.dumps(ExtractedPerson.model_json_schema(), indent=2))
print('\nParsedDate schema:')
print(json.dumps(ParsedDate.model_json_schema(), indent=2))

## Part 4: JSON Constrained Generation with outlines

The `outlines.generate.json()` generator accepts a Pydantic model and guarantees
the output will parse into it without errors.

In [ ]:
def generate_constrained_json(model_name: str, schema_model: type, prompt: str) -> dict:
    """
    Generate JSON constrained to a Pydantic schema using outlines.
    Falls back to a mock response if outlines/transformers not installed.
    """
    try:
        import outlines
        model = outlines.models.transformers(model_name)
        generator = outlines.generate.json(model, schema_model)
        result = generator(prompt)
        return result.model_dump()
    except (ImportError, Exception) as e:
        print(f"  [outlines not available: {type(e).__name__}] Returning mock output.")
        if schema_model is ExtractedPerson:
            return ExtractedPerson(name="Alice Chen", age=32, occupation="data scientist").model_dump()
        elif schema_model is ParsedDate:
            return ParsedDate(year=2024, month=3, day=15).model_dump()
        return {}

print('Function defined: generate_constrained_json')

In [ ]:
LOCAL_MODEL = "microsoft/phi-2"  # small model, ~2GB — requires GPU
person_prompt = "Extract: 'Dr. James Kim, 45, is a senior machine learning engineer at Acme AI.'"

print("Running outlines JSON constrained generation...")
outlines_person = generate_constrained_json(LOCAL_MODEL, ExtractedPerson, person_prompt)
print(f"Result: {outlines_person}")

# Validate it round-trips through Pydantic
parsed = ExtractedPerson(**outlines_person)
print(f"Pydantic validation: OK — name={parsed.name}, age={parsed.age} (type: {type(parsed.age).__name__})")

## Part 5: Regex Constrained Generation

Beyond JSON, outlines can constrain output to any regex pattern.
This is useful for phone numbers, dates, codes, IDs — any structured string.

In [ ]:
import re

def generate_constrained_regex(model_name: str, pattern: str, prompt: str) -> str:
    """Generate a string matching a regex pattern using outlines."""
    try:
        import outlines
        model = outlines.models.transformers(model_name)
        generator = outlines.generate.regex(model, pattern)
        return generator(prompt)
    except (ImportError, Exception) as e:
        print(f"  [outlines not available: {type(e).__name__}] Returning mock output.")
        mock = "555-867-5309"
        return mock if re.match(pattern, mock) else "000-000-0000"

PHONE_REGEX = r"\d{3}-\d{3}-\d{4}"
phone_prompt = "What is a typical US phone number format? Give me one example."

phone_result = generate_constrained_regex(LOCAL_MODEL, PHONE_REGEX, phone_prompt)
match = re.match(PHONE_REGEX, phone_result)
print(f"Generated: {phone_result}")
print(f"Matches pattern {PHONE_REGEX!r}: {match is not None}")

## Part 6: Choice Constrained Generation

`outlines.generate.choice()` restricts output to an exact list of strings.
Perfect for classification tasks — the output is guaranteed to be one of the choices.

In [ ]:
def generate_constrained_choice(model_name: str, choices: list, prompt: str) -> str:
    """Generate one of the given choices using outlines."""
    try:
        import outlines
        model = outlines.models.transformers(model_name)
        generator = outlines.generate.choice(model, choices)
        return generator(prompt)
    except (ImportError, Exception) as e:
        print(f"  [outlines not available: {type(e).__name__}] Returning mock output.")
        return choices[0]

SENTIMENT_CHOICES = ["positive", "negative", "neutral"]
sentiment_prompt = "The product was amazing and exceeded all my expectations! Sentiment:"

sentiment = generate_constrained_choice(LOCAL_MODEL, SENTIMENT_CHOICES, sentiment_prompt)
print(f"Sentiment: {sentiment!r}")
print(f"In valid choices: {sentiment in SENTIMENT_CHOICES}  ← always True with outlines")

## Part 7: instructor Comparison — Retry-Based Approach

instructor patches the OpenAI client to add Pydantic validation.
If the model returns invalid JSON, instructor sends the error back and retries.

In [ ]:
def extract_with_instructor(prompt: str, schema_model: type) -> dict:
    """Extract structured data using instructor (OpenAI tool-calling + Pydantic retry)."""
    try:
        import instructor
        from openai import OpenAI
        client = instructor.from_openai(OpenAI())
        result = client.chat.completions.create(
            model="gpt-4o-mini",
            response_model=schema_model,
            messages=[{"role": "user", "content": prompt}],
            max_retries=3,
        )
        return result.model_dump()
    except ImportError:
        # Fall back to direct OpenAI with JSON mode
        from openai import OpenAI
        client = OpenAI()
        schema_str = schema_model.model_json_schema()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": f"Extract data as JSON matching: {json.dumps(schema_str)}"},
                {"role": "user", "content": prompt},
            ],
        )
        return json.loads(resp.choices[0].message.content)

print('Function defined: extract_with_instructor')

In [ ]:
print("=== Side-by-side comparison: ExtractedPerson ===")
print(f"Prompt: {person_prompt}\n")

print("[outlines] CFG sampling (local model):")
print(f"  {outlines_person}")

print("\n[instructor] tool-calling + retry (gpt-4o-mini):")
instructor_person = extract_with_instructor(person_prompt, ExtractedPerson)
print(f"  {instructor_person}")

print("\nBoth produce valid ExtractedPerson objects — different mechanisms, same guarantee.")

## Part 8: When to Use Which Approach

### Use outlines when:
- Running **local models** (HuggingFace Transformers, llama.cpp)
- You need **mathematical guarantee** — no retries, no API errors possible
- Working with **regex or grammar constraints** (not just JSON)
- **Cost matters** — no extra API calls for retries
- Building **offline** or **edge** applications

### Use instructor when:
- Using **hosted APIs** (OpenAI, Anthropic, Groq)
- You need **complex nested schemas** with custom validators
- You already have an **OpenAI client** setup
- The model is **strong enough** to rarely need retries
- You want **streaming** structured output

In [ ]:
import time

print("=== Timing comparison ===")
print("Note: outlines timing includes model load on first call (cached after)")
print()

# Time instructor (gpt-4o-mini API call)
t0 = time.time()
instructor_result = extract_with_instructor(
    "Alice is 28 and works as a UX designer.", ExtractedPerson
)
instructor_time = time.time() - t0
print(f"instructor (gpt-4o-mini): {instructor_time:.2f}s — {instructor_result}")

# Time outlines (local model — fallback if not available)
t0 = time.time()
outlines_result = generate_constrained_json(
    LOCAL_MODEL, ExtractedPerson, "Alice is 28 and works as a UX designer."
)
outlines_time = time.time() - t0
print(f"outlines (local model):   {outlines_time:.2f}s — {outlines_result}")

print()
print("On GPU: outlines is typically faster after model load.")
print("On CPU with fallback: instructor wins on latency (network vs local inference).")

## Exercises

Practice what you've learned:

1. **New schema:** Create a `ProductReview` Pydantic model with fields: `product_name` (str), `rating` (int, 1–5), `sentiment` (str), `summary` (str). Use `generate_constrained_json()` to extract from a review text.

2. **Date regex:** Write a regex pattern for ISO dates (`YYYY-MM-DD`) and use `generate_constrained_regex()` to generate one. Verify with Python's `datetime.date.fromisoformat()`.

3. **Choice expansion:** Add `"mixed"` to `SENTIMENT_CHOICES` and classify 3 different product reviews. Check that outlines always returns a valid choice.

4. **Comparison:** For Exercise 1, compare `generate_constrained_json()` vs `extract_with_instructor()` on 5 different review texts. Which produces more consistent `rating` types (int vs string)?

In [ ]:
# Exercise 1: ProductReview schema
# Your code here

class ProductReview(BaseModel):
    product_name: str
    rating: int  # 1-5
    sentiment: str
    summary: str

review_text = "The Acme Blender Pro is fantastic! It blends smoothly and quietly. 5 stars."
# result = generate_constrained_json(LOCAL_MODEL, ProductReview, review_text)
# print(result)
print('ProductReview schema:', json.dumps(ProductReview.model_json_schema(), indent=2))

In [ ]:
# Exercise 2: ISO date regex
import datetime

DATE_REGEX = r"\d{4}-\d{2}-\d{2}"  # YYYY-MM-DD
date_prompt = "What is today's date in ISO format?"

date_result = generate_constrained_regex(LOCAL_MODEL, DATE_REGEX, date_prompt)
print(f"Generated date string: {date_result}")
print(f"Matches regex: {bool(re.match(DATE_REGEX, date_result))}")

# Try parsing it
try:
    parsed_date = datetime.date.fromisoformat(date_result)
    print(f"Parsed as date: {parsed_date}")
except ValueError as e:
    print(f"Note: valid regex but invalid calendar date — {e}")
    print("(outlines guarantees regex match, not calendar validity — add a validator for that)")

## Answer Key

### Exercise 1 — ProductReview
```python
class ProductReview(BaseModel):
    product_name: str
    rating: int
    sentiment: str
    summary: str

result = generate_constrained_json(LOCAL_MODEL, ProductReview, review_text)
parsed = ProductReview(**result)
assert isinstance(parsed.rating, int)  # outlines guarantees int, not "5"
```

### Exercise 2 — Date regex
```python
DATE_REGEX = r"\d{4}-\d{2}-\d{2}"
result = generate_constrained_regex(LOCAL_MODEL, DATE_REGEX, "Today's ISO date:")
# result will always match \d{4}-\d{2}-\d{2}
# For full calendar validity, use a Pydantic model with a date field instead
```

### Exercise 3 — Mixed sentiment
```python
CHOICES = ["positive", "negative", "neutral", "mixed"]
# outlines.generate.choice(model, CHOICES) always returns one of the 4 options
# instructor may return "somewhat positive" or other variants without cleanup
```

### Exercise 4 — Type consistency
outlines guarantees `rating` is an `int` (JSON integer) — the schema enforces it at the token level.
instructor with JSON mode usually returns int, but without instructor it can return `"5"` (string).

## Workshop Complete

**What you learned:**
- outlines uses CFG sampling to make invalid tokens impossible — one pass, guaranteed valid output
- instructor uses tool-calling + Pydantic retry — works with hosted APIs, may take multiple calls
- 3 constraint types: JSON schema, regex pattern, choice list
- When to choose each approach based on model type, latency, and schema complexity

**Next steps:**
- Try [example 120 — instructor structured extraction](../120-instructor-structured-extraction/) for deeper instructor patterns
- Explore outlines CFG grammars for even more complex constraints (e.g., valid SQL, arithmetic expressions)
- Run on a GPU (Colab T4 is free) to see outlines at full speed